# Fabric Database Hub hybrid SQL demo: deployment companion and validator

This notebook is the presenter's companion for a demo of **Microsoft Fabric Database Hub (preview)** and its **Database Agent (preview)** across three SQL targets in one Azure resource group:

| ID | Target | Hosting | How Database Hub sees it |
|---|---|---|---|
| T1 `sqlvm01` | SQL Server 2022 Standard | Azure VM with the SQL IaaS Agent extension | Estate through the extension; Performance through SQL VM performance monitoring |
| T2 `arcsql01` | SQL Server 2022 Standard | Nested Hyper-V guest on the Azure VM `archost01`, disclosed as a **simulated on-premises server** and enabled by Azure Arc | Estate through Azure Arc; Performance through Arc SQL monitoring |
| T3 `azuresql01-<deploymentId>` | Azure SQL Database `ContosoCare` | PaaS, General Purpose serverless | Estate natively; Performance through the `MS_EnablePerformanceMonitoringPreview` extended property |

**What this notebook does**

- **Deployment companion and architecture explainer:** explains what gets deployed, how, and why, and what it costs to leave running.
- **Validator:** checks the Fabric side with the Fabric REST API and the Hub side with the documented SQL telemetry endpoint, and records every check with the demo's status model: **Pass**, **Fail**, **Warning**, **Manual action required**, **Not supported**, or **Not selected**.
- **Presenter script, troubleshooting and cleanup guide:** the demo beats, the Hub and Database Agent prompts, remediation, reset, start and stop, and cleanup.

**What it doesn't do**

- It can't read or change Azure resources. Fabric notebooks can't get an Azure Resource Manager token, so starting, stopping, and deploying run from the PowerShell scripts or GitHub Actions. The companion notebook **`NB_DatabaseHub_LiveActivity`** can trigger GitHub Actions when the optional prerequisite P9 is in place.
- It never calls Database Hub private routes, never automates or scrapes the Hub UI, and never prints a token.

**How to run it**

1. Keep `LH_DatabaseHub` attached as the default lakehouse: the scripts' import binds the notebook to it. If you imported the notebook by hand, add `LH_DatabaseHub` in the Explorer pane and make it the default first.
2. Select **Run all**. That's safe: no cell changes Azure or Fabric resources, and the only destructive cell (cleanup, at the end) does nothing unless you set both `PURGE_DEMO_TABLES = True` and `CONFIRM = True`.
3. Read the result tables. Every result other than **Pass** comes with a remediation.
4. Results are appended to Delta tables in `LH_DatabaseHub`: `validation_results` and `telemetry_snapshots`.

> **Preview disclosure:** Database Hub, its Database Agent and Copilot experience, SQL Server on Azure VM performance monitoring, the Arc SQL monitoring setting, and the SQL telemetry endpoint are previews. Behavior, prerequisites, and APIs can change. This notebook uses documented interfaces only, plus one clearly labeled check of an undocumented path (a notebook `kusto` token against the telemetry endpoint) that falls back to documented steps.

**Contents:** 1 Parameters · 2 Overview and value · 3 Architecture · 4 Security and permissions · 5 Prerequisites check · 6 Cost reminders · 7 Deployment status · 8 SQL Server, Azure Arc, and Azure SQL · 9 Hub telemetry validation · 10 Hub discovery and per-target validation · 11 Agent skills and the Database Agent · 12 Optional mirroring · 13 Data generation and scenarios · 14 Hub investigation, agent prompts, and expected findings · 15 Remediation, reset, start and stop · 16 Validation summary · 17 Troubleshooting · 18 Cleanup · 19 Limitations and preview disclosures · 20 Official references · 21 Next steps

## 1. Parameters

The next cell is the notebook's **parameters cell** (in Fabric: **...** > **Toggle parameter cell**), so a pipeline notebook activity or `notebookutils.notebook.run` can override any value. It holds environment names and IDs only. **Never put a credential in it**: tokens come from `notebookutils.credentials` at run time.

The defaults match the repo's naming (`docs/architecture.md`): resource group `rg-<prefix>-<env>`, T1 `sqlvm01`, T2 `arcsql01`, and T3 server `azuresql01-<deploymentId>` with database `ContosoCare`.

In [ ]:
# Parameters cell (tagged "parameters"). Environment values only: never put a credential here.

# Fabric workspace and default lakehouse that hold the demo notebooks (prerequisite P5).
# The repo stores no Fabric IDs (ADR 0021): the scripts' import fills these two in. Empty means this session's
# workspace and default lakehouse, so a notebook you import by hand works too.
WORKSPACE_ID = ""
LAKEHOUSE_ID = ""
LAKEHOUSE_NAME = "LH_DatabaseHub"
LIVE_ACTIVITY_NOTEBOOK = "NB_DatabaseHub_LiveActivity"

# Demo environment names (docs/architecture.md).
RESOURCE_GROUP = "rg-dbhub-demo"
T1_VM_NAME = "sqlvm01"
T2_MACHINE_NAME = "arcsql01"
T3_SERVER_PREFIX = "azuresql01-"
DEMO_DATABASE = "ContosoCare"

# Targets in this demo, and the T2 install mode (t2SqlInstallMode). Developer mode has no Hub Performance on T2.
TARGETS = "T1,T2,T3"
T2_SQL_INSTALL_MODE = "PaygWizard"

# Telemetry checks: how far back to look, and how recent the last sample must be to count as reporting.
TELEMETRY_LOOKBACK_MINUTES = 60
TELEMETRY_FRESH_MINUTES = 15

# The repo, for links to docs and workflows; the scripts' import sets it from github.repository when it's set.
GITHUB_REPOSITORY = "MarcBushong/fabric-database-hub-demo"

# Append results to the default lakehouse's Delta tables (validation_results, telemetry_snapshots).
SAVE_RESULTS = True

# Safety switches for the only resource-changing cell in this notebook (cleanup, section 18): it drops the demo's
# Delta tables only when both are True. Leave them False.
PURGE_DEMO_TABLES = False
CONFIRM = False

## 2. Overview and value

**The story in four beats** (the full 15- and 30-minute scripts are in `docs/demo-walkthrough.md`):

1. **Before:** fragmented visibility. An Azure VM, an on-premises server, and a PaaS database each have their own portal, metrics, and tools, so an investigation starts with tool switching.
2. **After:** Database Hub shows SQL Server on an Azure VM (T1), an Arc-enabled "on-premises" SQL Server (T2, a nested guest disclosed as simulated), and Azure SQL Database (T3) in one experience, within what the preview documents.
3. **Incident:** baseline, then a bounded locking workload, then the affected database, the telemetry and findings, the Database Agent's investigation, evidence and root cause, a recommended action, **human approval**, stop or remediate, and verified recovery.
4. **Value:** faster detection, less tool switching, a consistent hybrid inventory, shared context, guided investigation, and human-controlled action.

**What the customer sees in the Hub**

| Hub page | What it shows in this demo |
|---|---|
| **Overview** | **What needs attention?** issues and suggestions across the estate, plus CPU, memory, storage, and IO summaries |
| **Estate** | One inventory of T1, T2, and T3, with issue and suggestion chips under each resource |
| **Performance** | The Microsoft SQL dashboard: CPU, memory, storage, IO, and workload pressure for the selected servers |
| **Database Agent and Copilot** | Investigation of findings with an auditable proposal that a human approves before anything runs |

Database Hub is free with a Fabric Free license and uses each person's own Azure permissions. Nothing moves into Fabric: the databases stay where they are.

## 3. Architecture

Everything runs in **one Azure resource group** (`rg-dbhub-demo` by default) plus this Fabric workspace. The full design and its decision records are in `docs/architecture.md` and `docs/adr/`.

| Layer | Components |
|---|---|
| Network | One VNet (`10.42.0.0/24`), no VM public IPs, NSGs that deny inbound internet traffic, Bastion Basic only on demand |
| T1 | `sqlvm01`: SQL Server 2022 Standard pay-as-you-go image, SQL IaaS Agent extension, system-assigned identity. It also runs the workload runner for T3. **Never Arc-enabled.** |
| T2 | `archost01`: Hyper-V host with an internal NAT. Its guest `arcsql01` runs Windows Server 2022 Evaluation and SQL Server 2022 Standard, with the Azure Connected Machine agent and the Azure extension for SQL Server **in the guest only**. The guest can't reach the host's instance metadata service. |
| T3 | Logical server `azuresql01-<deploymentId>` and database `ContosoCare`: General Purpose serverless, Microsoft Entra-only authentication, private endpoint where policy requires it |
| Signals | A Microsoft-managed telemetry pipeline (free) feeds Database Hub **Performance** and the read-only telemetry endpoint (`ArcSqlTelemetry`, up to 7 days). Azure Resource Graph feeds Hub **Estate**. |
| Control | `scripts/*.ps1` (PowerShell 7, Contributor) and GitHub Actions (optional, P7) start, stop, reset, and deploy. Notebooks can't call Azure Resource Manager; the live-activity notebook can trigger `demo-control.yml` (optional, P9). |
| This workspace | `NB_DatabaseHub_Deployment` (this notebook), `NB_DatabaseHub_LiveActivity`, and `LH_DatabaseHub` with the Delta tables `validation_results`, `scenario_runs`, `telemetry_snapshots`, and `analysis_findings` |

**Data flow:** SQL targets → telemetry pipeline → Database Hub Performance, and the telemetry endpoint → these notebooks (with a fallback to the Azure Data Explorer web UI). **Inventory flow:** Azure Resource Manager → Azure Resource Graph → Database Hub Estate. **Control flow:** operator or GitHub Actions → Azure Resource Manager and managed Run Command → the VMs.

## 4. Security and permissions

| Who or what | Needs | Why |
|---|---|---|
| Deployment operator | **Contributor** on the subscription (P1) | Deploys and configures everything; creates no role assignments by default |
| Presenter in Database Hub | **Reader** on the subscription (Contributors already have it; P8 otherwise), `Microsoft.Sql` registered (P2), the Hub tenant setting (P4), and a Fabric Free license | Database Hub and the telemetry endpoint show only what the signed-in identity can read |
| Notebook user | **Contributor** or higher on this workspace, which is on F or Trial capacity (P5) | Run notebooks and write the lakehouse tables |
| Live-activity triggering (optional) | A GitHub fine-grained token with **Actions: write** on this repo only, in a Key Vault secret the notebook user can read (P9) | `workflow_dispatch` of `demo-control.yml` |

**How this notebook handles credentials**

- It holds **no credentials**. It asks `notebookutils.credentials.getToken` for tokens at run time, using only the documented audience keys: `pbi` for the Fabric REST API and `kusto` for the telemetry endpoint. A token lives in one HTTP header for one request; it's never printed, logged, returned, or saved.
- Error text is passed through a redaction filter before it's printed.
- The committed notebook has **no outputs**: CI fails on outputs or execution counts, and a pre-commit hook strips them.
- T3 is Microsoft Entra-only, no SQL Server has a public endpoint by default, and any action that the Database Agent proposes needs **human approval** before it runs.

### Helpers

Run the next cell before any other code cell. It validates the parameters and defines the helpers for tokens, the Fabric REST API, telemetry queries, results, and Delta tables.

- **Tokens:** `notebookutils.credentials.getToken` with the documented audience keys only (`storage`, `pbi`, `keyvault`, and `kusto` exist; this notebook uses `pbi` and `kusto`). There's no Azure Resource Manager audience.
- **Telemetry:** the documented REST endpoint (`POST https://adx.centralus.arcdataservices.com/kusto/v1/rest/query`, database `ArcSqlTelemetry`). Its documented token audience is `https://kusto.kusto.windows.net`; whether it accepts a notebook `kusto` token **isn't documented**, so section 9 checks it, and every query falls back to printing its KQL with the documented Azure Data Explorer web UI steps.
- **KQL rules** from the telemetry article: filter on time first, take deltas of cumulative counters (never sum them), branch on `CounterType`, group by `ResourceID` and `SqlServerInstanceName`, and compare resource names case-insensitively.

In [ ]:
# Helpers: parameters check, tokens, Fabric REST, telemetry queries, results, and Delta tables. Run this cell first.
# Tokens come from notebookutils.credentials.getToken with documented audience keys only. A token is used in one
# HTTP header and never printed, logged, returned, or saved.
import datetime as dt
import json
import re
import time
import uuid

import pandas as pd
import requests

NOTEBOOK_NAME = "NB_DatabaseHub_Deployment"
FABRIC_API = "https://api.fabric.microsoft.com/v1"
TELEMETRY_QUERY_URL = "https://adx.centralus.arcdataservices.com/kusto/v1/rest/query"
TELEMETRY_CONNECTION_URI = "https://adx.centralus.arcdataservices.com/kusto/"
TELEMETRY_DATABASE = "ArcSqlTelemetry"
ADX_WEB_UI = "https://dataexplorer.azure.com/"
DATABASE_HUB_URL = "https://powerbi.com/workloads/fdh/databaseHub"
REPO_URL = "https://github.com/" + GITHUB_REPOSITORY
# notebookutils.credentials.getToken supports only these audience keys. None of them is Azure Resource Manager.
ALLOWED_TOKEN_AUDIENCES = ("storage", "pbi", "keyvault", "kusto")
STATUSES = ("Pass", "Fail", "Warning", "Manual action required", "Not supported", "Not selected")
TARGET_NAMES = {"T1": "T1 " + T1_VM_NAME, "T2": "T2 " + T2_MACHINE_NAME, "T3": "T3 " + T3_SERVER_PREFIX + "<id>/" + DEMO_DATABASE}
DOCS = {
    "notebookutils": "https://learn.microsoft.com/fabric/data-engineering/notebookutils/notebookutils-credentials",
    "runtime": "https://learn.microsoft.com/fabric/data-engineering/notebookutils/notebookutils-runtime",
    "telemetry": "https://learn.microsoft.com/fabric/database/hub/microsoft-sql-query-performance-monitoring-telemetry",
    "monitor": "https://learn.microsoft.com/fabric/database/hub/monitor-sql",
    "addSql": "https://learn.microsoft.com/fabric/database/hub/add-sql",
    "licenses": "https://learn.microsoft.com/fabric/enterprise/licenses",
    "getWorkspace": "https://learn.microsoft.com/rest/api/fabric/core/workspaces/get-workspace",
    "getLakehouse": "https://learn.microsoft.com/rest/api/fabric/lakehouse/items/get-lakehouse",
    "listItems": "https://learn.microsoft.com/rest/api/fabric/core/items/list-items",
}
TELEMETRY_HTTP_HINTS = {
    400: "HTTP 400: the KQL has an error, or a table or column changed during the preview (check names with getschema).",
    401: "HTTP 401: the endpoint didn't accept the notebook's kusto token. A notebook kusto token isn't documented for this endpoint.",
    403: "HTTP 403: the signed-in user needs Reader on the subscription, and Microsoft.Sql must be registered on it.",
    500: "HTTP 500: the request body wasn't valid.",
}
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
RESULTS = []
COVERAGE = None
COVERAGE_WINDOW = {"start": None, "end": None}
TELEMETRY_STATE = {"checked": False, "available": False, "message": "The telemetry access check (section 10) hasn't run yet."}


def _require(pattern, value, name):
    if not isinstance(value, str) or not re.fullmatch(pattern, value):
        raise ValueError("Parameter %s has an unexpected value: %r" % (name, value))
    return value


def _flag(value):
    """Parameters can arrive as strings from a pipeline; only an explicit true value counts as True."""
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in ("true", "1", "yes")


CONFIRM = _flag(CONFIRM)
PURGE_DEMO_TABLES = _flag(PURGE_DEMO_TABLES)
SAVE_RESULTS = _flag(SAVE_RESULTS)


_GUID = r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}"
# Empty IDs are resolved from the session once runtime_context() is defined, below (ADR 0021).
WORKSPACE_ID = str(WORKSPACE_ID or "").strip()
LAKEHOUSE_ID = str(LAKEHOUSE_ID or "").strip()
_require(r"[A-Za-z0-9 _.-]{1,256}", LAKEHOUSE_NAME, "LAKEHOUSE_NAME")
_require(r"[A-Za-z0-9 _.-]{1,256}", LIVE_ACTIVITY_NOTEBOOK, "LIVE_ACTIVITY_NOTEBOOK")
# Names are checked against strict patterns before they're placed in KQL.
_require(r"[A-Za-z0-9._()-]{1,90}", RESOURCE_GROUP, "RESOURCE_GROUP")
_require(r"[A-Za-z0-9-]{1,64}", T1_VM_NAME, "T1_VM_NAME")
_require(r"[A-Za-z0-9._-]{1,64}", T2_MACHINE_NAME, "T2_MACHINE_NAME")
_require(r"[a-z0-9-]{1,63}", T3_SERVER_PREFIX, "T3_SERVER_PREFIX")
_require(r"[A-Za-z0-9_]{1,128}", DEMO_DATABASE, "DEMO_DATABASE")
_require(r"[A-Za-z0-9._-]+/[A-Za-z0-9._-]+", GITHUB_REPOSITORY, "GITHUB_REPOSITORY")
_require(r"PaygWizard|ProductKey|Developer", T2_SQL_INSTALL_MODE, "T2_SQL_INSTALL_MODE")
SELECTED_TARGETS = [item.strip().upper() for item in str(TARGETS).split(",") if item.strip()]
if not SELECTED_TARGETS or any(item not in ("T1", "T2", "T3") for item in SELECTED_TARGETS):
    raise ValueError("TARGETS must list T1, T2, and/or T3, for example 'T1,T2,T3'.")
if not 5 <= int(TELEMETRY_LOOKBACK_MINUTES) <= 7 * 24 * 60:
    raise ValueError("TELEMETRY_LOOKBACK_MINUTES must be between 5 minutes and the 7-day retention.")
if not 1 <= int(TELEMETRY_FRESH_MINUTES) <= int(TELEMETRY_LOOKBACK_MINUTES):
    raise ValueError("TELEMETRY_FRESH_MINUTES must be between 1 and TELEMETRY_LOOKBACK_MINUTES.")

_REDACTIONS = (
    (re.compile(r"eyJ[A-Za-z0-9_-]{5,}\.[A-Za-z0-9_-]{5,}\.[A-Za-z0-9_-]*"), "[REDACTED:jwt]"),
    (re.compile(r"(?i)(bearer\s+)[A-Za-z0-9._~+/=-]{8,}"), r"\1[REDACTED]"),
    (re.compile(r"\b(?:gh[pousr]_[A-Za-z0-9]{20,}|github_pat_[A-Za-z0-9_]{20,})\b"), "[REDACTED:github-token]"),
    (re.compile(r"(?i)((?:^|[?&;\s\"'])sig=)[^&\s\"']+"), r"\1[REDACTED]"),
)


def redact(value):
    """Removes token-like strings from text before it's printed or saved."""
    text = "" if value is None else str(value)
    for pattern, replacement in _REDACTIONS:
        text = pattern.sub(replacement, text)
    return text


def utc_now():
    return dt.datetime.now(dt.timezone.utc)


def _notebookutils():
    try:
        return notebookutils  # noqa: F821 - Fabric provides notebookutils in every notebook session.
    except NameError:
        try:
            import notebookutils as module
            return module
        except ImportError:
            return None


def _spark():
    try:
        return spark  # noqa: F821 - Fabric provides the Spark session.
    except NameError:
        return None


def display_frame(frame):
    try:
        display(frame)  # noqa: F821 - Fabric provides display().
    except NameError:
        print(frame.to_string(index=False))


def get_token(audience):
    """Returns a token for a documented notebookutils audience key. Never print, log, or save the result."""
    if audience not in ALLOWED_TOKEN_AUDIENCES:
        raise ValueError("notebookutils.credentials.getToken supports only %s." % ", ".join(ALLOWED_TOKEN_AUDIENCES))
    utils = _notebookutils()
    if utils is None:
        raise RuntimeError("notebookutils isn't available. Run this notebook in Microsoft Fabric.")
    return utils.credentials.getToken(audience)


def runtime_context():
    utils = _notebookutils()
    if utils is None:
        return {}
    try:
        return dict(utils.runtime.context or {})
    except Exception:
        return {}


# The repo stores no Fabric IDs (ADR 0021). Empty IDs, as in a notebook imported by hand, are this session's own
# workspace and default lakehouse; the scripts' import fills in the ones it resolved by name.
if not WORKSPACE_ID or not LAKEHOUSE_ID:
    _session = runtime_context()
    WORKSPACE_ID = WORKSPACE_ID or str(_session.get("currentWorkspaceId") or "")
    LAKEHOUSE_ID = LAKEHOUSE_ID or str(_session.get("defaultLakehouseId") or "")
if WORKSPACE_ID:
    _require(_GUID, WORKSPACE_ID, "WORKSPACE_ID")
if LAKEHOUSE_ID:
    _require(_GUID, LAKEHOUSE_ID, "LAKEHOUSE_ID")


def _retry_after(response):
    try:
        return min(max(int(response.headers.get("Retry-After", "5")), 1), 60)
    except ValueError:
        return 5


def _error_text(response):
    try:
        body = response.json()
    except ValueError:
        return redact((response.text or "")[:300])
    if isinstance(body, dict):
        parts = [str(body.get(key)) for key in ("errorCode", "message", "requestId") if body.get(key)]
        if isinstance(body.get("error"), dict):
            parts += [str(body["error"].get(key)) for key in ("code", "message") if body["error"].get(key)]
        return redact(" ".join(parts)[:300])
    return ""


def fabric_get(path_or_url):
    """GET from the Fabric REST API with the notebook user's pbi token. Returns (status, body, error)."""
    url = path_or_url if path_or_url.startswith(FABRIC_API + "/") else FABRIC_API + path_or_url
    if not url.startswith(FABRIC_API + "/"):
        raise ValueError("Refusing to send a Fabric token outside " + FABRIC_API)
    response = None
    for attempt in range(4):
        try:
            response = requests.get(url, headers={"Authorization": "Bearer " + get_token("pbi")}, timeout=60)
        except requests.RequestException as error:
            return 0, None, redact(error)
        except Exception as error:
            return 0, None, "notebookutils didn't return a pbi token: " + redact(error)
        if response.status_code == 429 and attempt < 3:
            time.sleep(_retry_after(response))
            continue
        break
    if response.ok:
        try:
            return response.status_code, response.json(), ""
        except ValueError:
            return response.status_code, None, "The response wasn't JSON."
    return response.status_code, None, _error_text(response)


def fabric_list_items(workspace_id, item_type):
    """Lists items of one type in a workspace, following continuation pages. Returns (items, error)."""
    items = []
    url = "%s/workspaces/%s/items?type=%s" % (FABRIC_API, workspace_id, item_type)
    for _ in range(50):
        if not url:
            return items, ""
        status, body, error = fabric_get(url)
        if status != 200 or body is None:
            return None, "HTTP %s %s" % (status, error)
        # A continuation URI can drop the type filter, so keep only items of the requested type.
        items.extend(item for item in body.get("value", []) if item.get("type") in (None, item_type))
        url = body.get("continuationUri")
        if url and not url.startswith(FABRIC_API + "/"):
            return None, "Unexpected continuation URI; stopped so the token stays inside the Fabric API."
    return (items, "") if not url else (None, "Stopped after 50 pages of results.")


def kql_string(value):
    return "'" + str(value).replace("\\", "\\\\").replace("'", "\\'") + "'"


def kql_datetime(value):
    return "datetime(" + value.astimezone(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ") + ")"


def kql_prelude(start, end):
    """Time window, names, and a function that maps telemetry rows to T1, T2, or T3 (empty for anything else)."""
    return "\n".join([
        "let _start = %s;" % kql_datetime(start),
        "let _end = %s;" % kql_datetime(end),
        "let _rg = %s;" % kql_string(RESOURCE_GROUP),
        "let _t1 = %s;" % kql_string(T1_VM_NAME),
        "let _t2 = %s;" % kql_string(T2_MACHINE_NAME),
        "let _t3prefix = %s;" % kql_string(T3_SERVER_PREFIX),
        "let _t3suffix = %s;" % kql_string("/" + DEMO_DATABASE),
        "let _target = (resourceType: string, resourceName: string) {",
        "    case(resourceType == 'virtualMachines' and resourceName =~ _t1, 'T1',",
        "         resourceType == 'machines' and resourceName =~ _t2, 'T2',",
        "         resourceType == 'servers/databases' and resourceName startswith _t3prefix and resourceName endswith _t3suffix, 'T3',",
        "         '')",
        "};",
    ])


def _to_utc(series):
    try:
        return pd.to_datetime(series, utc=True, format="ISO8601")
    except (TypeError, ValueError):
        return pd.to_datetime(series, utc=True, errors="coerce")


def kusto_frame(body):
    """Turns a Kusto v1 response (Tables[0].Columns and Rows) into a DataFrame, mapping values by column name."""
    tables = body.get("Tables") or []
    if not tables:
        return pd.DataFrame()
    first = tables[0]
    columns = first.get("Columns", [])
    frame = pd.DataFrame(first.get("Rows", []), columns=[column.get("ColumnName") for column in columns])
    for column in columns:
        if str(column.get("ColumnType") or column.get("DataType")).lower() in ("datetime", "system.datetime"):
            frame[column["ColumnName"]] = _to_utc(frame[column["ColumnName"]])
    return frame


def _post_kql(kql):
    """Sends read-only KQL to the telemetry endpoint. Returns (status, frame, message)."""
    try:
        token = get_token("kusto")
    except Exception as error:
        return 0, None, "notebookutils didn't return a kusto token: " + redact(error)
    try:
        response = requests.post(
            TELEMETRY_QUERY_URL,
            json={"db": TELEMETRY_DATABASE, "csl": str(kql)},
            headers={"Authorization": "Bearer " + token, "Content-Type": "application/json"},
            timeout=180,
        )
    except requests.RequestException as error:
        return 0, None, "The telemetry endpoint couldn't be reached: " + redact(error)
    finally:
        token = None
    if response.status_code == 200:
        try:
            return 200, kusto_frame(response.json()), ""
        except ValueError:
            return 0, None, "The telemetry endpoint returned something other than Kusto v1 JSON."
    hint = TELEMETRY_HTTP_HINTS.get(response.status_code, "HTTP %d." % response.status_code)
    return response.status_code, None, (hint + " " + _error_text(response)).strip()


def print_kql_fallback(kql, title, reason):
    print("[%s] Not queried from the notebook: %s" % (title, redact(reason)))
    print("Run the KQL below yourself with the documented Azure Data Explorer web UI:")
    print("  1. Open %s and sign in with an account that can read the demo subscription." % ADX_WEB_UI)
    print("  2. Select Add > Connection, and enter the connection URI %s" % TELEMETRY_CONNECTION_URI)
    print("  3. Expand the connection, select the %s database, paste the query, and select Run." % TELEMETRY_DATABASE)
    print("  Or run ./scripts/validate.ps1 from the repo, which uses an Azure CLI token as the telemetry article documents.")
    print("----- KQL: %s -----" % title)
    print(kql)
    print("----- end of KQL -----")


def query_telemetry(kql, title):
    """Runs KQL with the notebook's kusto token. Returns a DataFrame, or None after printing the fallback."""
    if TELEMETRY_STATE["checked"] and not TELEMETRY_STATE["available"]:
        print_kql_fallback(kql, title, TELEMETRY_STATE["message"])
        return None
    status, frame, message = _post_kql(kql)
    if status == 200:
        return frame
    print_kql_fallback(kql, title, message)
    return None


def add_result(check_id, area, target, status, message, evidence=None, remediation="", doc_url=""):
    """Records one check with the ADR 0012 status model. Re-running a check replaces its earlier result."""
    if status not in STATUSES:
        raise ValueError("Unknown status: " + status)
    if not re.fullmatch(r"[A-Z0-9]+(\.[A-Z0-9_]+)+", check_id):
        raise ValueError("Invalid check ID: " + check_id)
    record = {
        "run_id": RUN_ID,
        "source_notebook": NOTEBOOK_NAME,
        "check_id": check_id,
        "area": area,
        "target": target,
        "status": status,
        "message": redact(message),
        "evidence": redact(json.dumps(evidence or {}, default=str, sort_keys=True)),
        "remediation": redact(remediation),
        "doc_url": doc_url,
        "evaluation_only": False,
        "timestamp_utc": utc_now(),
    }
    RESULTS[:] = [item for item in RESULTS if item["check_id"] != check_id]
    RESULTS.append(record)
    print("%-24s %s: %s" % (status, check_id, record["message"]))
    return record


def show_results(records=None):
    frame = pd.DataFrame(RESULTS if records is None else records)
    if frame.empty:
        print("No results yet.")
        return frame
    display_frame(frame[["status", "check_id", "target", "message", "remediation"]])
    counts = frame["status"].value_counts()
    print("Summary: " + ", ".join("%s %d" % (status, counts[status]) for status in STATUSES if status in counts))
    return frame


TABLE_COLUMNS = {
    "validation_results": [
        ("run_id", "string"), ("source_notebook", "string"), ("check_id", "string"), ("area", "string"),
        ("target", "string"), ("status", "string"), ("message", "string"), ("evidence", "string"),
        ("remediation", "string"), ("doc_url", "string"), ("evaluation_only", "boolean"), ("timestamp_utc", "timestamp"),
        ("report_date_utc", "date"),
    ],
    "telemetry_snapshots": [
        ("run_id", "string"), ("source_notebook", "string"), ("captured_at_utc", "timestamp"), ("analysis", "string"),
        ("target", "string"), ("resource_type", "string"), ("resource_name", "string"), ("instance_name", "string"),
        ("database_name", "string"), ("metric", "string"), ("bucket_utc", "timestamp"), ("value", "double"),
        ("unit", "string"), ("window_start_utc", "timestamp"), ("window_end_utc", "timestamp"), ("report_date_utc", "date"),
    ],
}


# The UTC calendar date of each row's primary time (the first one that isn't null). The Power BI history report
# (fabric/powerbi) relates every table to its Date table through this column.
REPORT_DATE_SOURCES = {
    "validation_results": ("timestamp_utc",),
    "scenario_runs": ("window_start_utc", "requested_at_utc"),
    "telemetry_snapshots": ("bucket_utc", "window_end_utc", "captured_at_utc"),
    "analysis_findings": ("window_start_utc", "captured_at_utc"),
    "incident_windows": ("window_start_utc",),
}


def _utc_date(value):
    """UTC calendar date of a timestamp; naive values are already UTC."""
    if value is None:
        return None
    stamp = pd.Timestamp(value)
    if pd.isna(stamp):
        return None
    return (stamp.tz_convert("UTC") if stamp.tzinfo is not None else stamp).date()


def report_date(table, row):
    """report_date_utc for one row: the UTC date of the first primary time in REPORT_DATE_SOURCES that isn't null."""
    for name in REPORT_DATE_SOURCES.get(table, ()):
        value = _utc_date(row.get(name))
        if value is not None:
            return value
    return None


def _struct(table):
    from pyspark.sql import types as spark_types
    mapping = {
        "string": spark_types.StringType(), "boolean": spark_types.BooleanType(), "timestamp": spark_types.TimestampType(),
        "double": spark_types.DoubleType(), "long": spark_types.LongType(), "integer": spark_types.IntegerType(),
        "date": spark_types.DateType(),
    }
    return spark_types.StructType([spark_types.StructField(name, mapping[kind], True) for name, kind in TABLE_COLUMNS[table]])


def _spark_value(value):
    if value is None:
        return None
    if isinstance(value, pd.Timestamp):
        return None if pd.isna(value) else value.to_pydatetime(warn=False)
    if isinstance(value, float) and value != value:
        return None
    if hasattr(value, "item") and not isinstance(value, (str, bytes)):
        return value.item()
    return value


def _rows_frame(session, table, rows):
    """A Spark DataFrame with the TABLE_COLUMNS schema; report_date_utc comes from REPORT_DATE_SOURCES."""
    names = [name for name, _ in TABLE_COLUMNS[table]]
    data = [tuple(_spark_value(report_date(table, row) if name == "report_date_utc" else row.get(name)) for name in names)
            for row in rows]
    return session.createDataFrame(data, schema=_struct(table))


def write_delta(table, rows, replace_run=True):
    """Appends rows to a Delta table in the default lakehouse. Lakehouse data only: no Azure or Fabric resource
    changes. With replace_run, rows from this RUN_ID are deleted first, so re-running a cell doesn't duplicate them."""
    if not rows:
        print("%s: nothing to save." % table)
        return 0
    session = _spark()
    if session is None:
        print("%s: Spark isn't available, so nothing was saved." % table)
        return 0
    if not runtime_context().get("defaultLakehouseId"):
        print("%s: no default lakehouse is attached, so nothing was saved. Attach %s and run the cell again." % (table, LAKEHOUSE_NAME))
        return 0
    frame = _rows_frame(session, table, rows)
    if replace_run and session.catalog.tableExists(table):
        session.sql("DELETE FROM %s WHERE run_id = '%s'" % (table, RUN_ID))
    # One Parquet file per write keeps every table far below Direct Lake's limit of 1,000 files per table (F16).
    frame.coalesce(1).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(table)
    print("%s: saved %d row(s) for run %s." % (table, len(rows), RUN_ID))
    return len(rows)


print("Helpers ready. Run ID: %s. Targets: %s." % (RUN_ID, ", ".join(SELECTED_TARGETS)))

## 5. Prerequisites check

The demo needs **Contributor on an existing subscription** and nothing more for the core deployment. `./scripts/preflight.ps1` checks the Azure-side items; this notebook checks what it can see from Fabric.

| # | Prerequisite | Checked here? |
|---|---|---|
| P1 | Contributor on the subscription | No: Azure Resource Manager (run `preflight.ps1`) |
| P2 | Registered resource providers, including `Microsoft.Sql` | Indirectly: telemetry queries fail with HTTP 403 without `Microsoft.Sql` (section 9) |
| P3 | Regional vCPU and public IP quota | No: `preflight.ps1` |
| P4 | Database Hub tenant setting **Users can access the Database hub (preview)**, and a Fabric Free license per Hub user | **Manual action required**: no documented API reads it without Fabric administrator rights |
| P5 | Contributor or higher on this workspace, on F or Trial capacity | **Yes**: Fabric REST API (workspace, capacity, lakehouse, notebooks) |
| P6 | An Entra group the deploying user owns, used as the T3 admin | No: `preflight.ps1` |
| P7 | GitHub one-click (optional) | No: `setup-github-oidc.ps1` |
| P8 | Subscription Reader for presenters who aren't Contributors (optional) | Indirectly: section 9 |
| P9 | GitHub token in Key Vault for the live-activity notebook (optional) | In `NB_DatabaseHub_LiveActivity` |
| P10 | SQL Server Standard product key (optional) | No |

The next cell reads the session context with `notebookutils.runtime.context`, then calls three documented Fabric REST APIs with the notebook user's `pbi` token: **Workspaces - Get Workspace**, **Items - Get Lakehouse**, and **Items - List Items**. It changes nothing.

In [ ]:
# Prerequisites check: Fabric session, workspace capacity (P5), lakehouse, notebooks, and the Hub tenant setting (P4). Read-only.
context = runtime_context()
if not context:
    add_result("FABRIC.SESSION.CONTEXT", "fabric", "fabric", "Not supported",
               "notebookutils isn't available, so the session context can't be read. Run this notebook in Microsoft Fabric.",
               doc_url=DOCS["runtime"])
else:
    current_workspace = context.get("currentWorkspaceId")
    if current_workspace == WORKSPACE_ID:
        add_result("FABRIC.SESSION.WORKSPACE", "fabric", "fabric", "Pass",
                   "This notebook runs in the demo workspace.", evidence={"workspaceId": current_workspace})
    else:
        add_result("FABRIC.SESSION.WORKSPACE", "fabric", "fabric", "Warning",
                   "This notebook runs in a different workspace than WORKSPACE_ID.",
                   evidence={"currentWorkspaceId": current_workspace, "expected": WORKSPACE_ID},
                   remediation="Run it from the demo workspace, or set WORKSPACE_ID and LAKEHOUSE_ID to this workspace.",
                   doc_url=DOCS["runtime"])
    default_lakehouse = context.get("defaultLakehouseId")
    if default_lakehouse == LAKEHOUSE_ID:
        add_result("FABRIC.SESSION.LAKEHOUSE", "fabric", "fabric", "Pass",
                   "%s is the default lakehouse, so results can be saved." % LAKEHOUSE_NAME,
                   evidence={"defaultLakehouseId": default_lakehouse, "defaultLakehouseName": context.get("defaultLakehouseName")})
    elif default_lakehouse:
        add_result("FABRIC.SESSION.LAKEHOUSE", "fabric", "fabric", "Warning",
                   "The default lakehouse isn't %s; results go to %s." % (LAKEHOUSE_NAME, context.get("defaultLakehouseName")),
                   evidence={"defaultLakehouseId": default_lakehouse}, remediation="Attach %s as the default lakehouse." % LAKEHOUSE_NAME)
    else:
        add_result("FABRIC.SESSION.LAKEHOUSE", "fabric", "fabric", "Fail",
                   "No default lakehouse is attached, so results can't be saved.",
                   remediation="In the Explorer pane, add %s and set it as the default lakehouse." % LAKEHOUSE_NAME)

status, workspace, error = fabric_get("/workspaces/%s" % WORKSPACE_ID)
workspace = workspace or {}
if status == 200:
    capacity_id = workspace.get("capacityId")
    progress = workspace.get("capacityAssignmentProgress")
    evidence = {"workspace": workspace.get("displayName"), "capacityId": capacity_id,
                "capacityRegion": workspace.get("capacityRegion"), "capacityAssignmentProgress": progress}
    if capacity_id and progress in (None, "Completed"):
        add_result("FABRIC.P5.CAPACITY", "fabric", "fabric", "Pass",
                   "The workspace is on a capacity, so notebooks can run (P5).", evidence=evidence, doc_url=DOCS["getWorkspace"])
    elif capacity_id:
        add_result("FABRIC.P5.CAPACITY", "fabric", "fabric", "Warning",
                   "The workspace's capacity assignment is %s." % progress, evidence=evidence,
                   remediation="Wait for the capacity assignment to complete, then run this cell again.", doc_url=DOCS["getWorkspace"])
    else:
        add_result("FABRIC.P5.CAPACITY", "fabric", "fabric", "Fail",
                   "The workspace isn't on a capacity. Notebooks need F or Trial capacity, even though Database Hub is free.",
                   evidence=evidence, remediation="Workspace settings > License info: assign an F or Trial capacity (P5).",
                   doc_url=DOCS["licenses"])
elif status in (401, 403, 404):
    add_result("FABRIC.P5.CAPACITY", "fabric", "fabric", "Fail",
               "The workspace can't be read (HTTP %s): %s" % (status, error),
               remediation="Ask a workspace admin for Contributor or higher on the workspace (P5), and check WORKSPACE_ID.",
               doc_url=DOCS["getWorkspace"])
else:
    add_result("FABRIC.P5.CAPACITY", "fabric", "fabric", "Warning",
               "The workspace check didn't complete (HTTP %s): %s" % (status, error),
               remediation="Run this cell again in a minute.", doc_url=DOCS["getWorkspace"])

if LAKEHOUSE_ID:
    status, lakehouse, error = fabric_get("/workspaces/%s/lakehouses/%s" % (WORKSPACE_ID, LAKEHOUSE_ID))
else:
    status, lakehouse, error = 404, None, "LAKEHOUSE_ID is empty and no default lakehouse is attached"
lakehouse = lakehouse or {}
if status == 200 and lakehouse.get("displayName") == LAKEHOUSE_NAME:
    add_result("FABRIC.LAKEHOUSE.EXISTS", "fabric", "fabric", "Pass",
               "%s exists in the workspace." % LAKEHOUSE_NAME, evidence={"lakehouseId": LAKEHOUSE_ID}, doc_url=DOCS["getLakehouse"])
elif status == 200:
    add_result("FABRIC.LAKEHOUSE.EXISTS", "fabric", "fabric", "Warning",
               "Lakehouse %s is named %s, not %s." % (LAKEHOUSE_ID, lakehouse.get("displayName"), LAKEHOUSE_NAME),
               remediation="Set LAKEHOUSE_NAME to match, or bind the notebook to the right lakehouse.", doc_url=DOCS["getLakehouse"])
else:
    add_result("FABRIC.LAKEHOUSE.EXISTS", "fabric", "fabric", "Fail",
               "Lakehouse %s wasn't found or can't be read (HTTP %s): %s" % (LAKEHOUSE_ID or LAKEHOUSE_NAME, status, error),
               remediation="Run ./scripts/configure-fabric.ps1, which creates %s when it's missing and binds the notebooks to it (ADR 0021); or attach it as the default lakehouse." % LAKEHOUSE_NAME,
               doc_url=DOCS["getLakehouse"])

notebooks, error = fabric_list_items(WORKSPACE_ID, "Notebook")
if notebooks is None:
    add_result("FABRIC.NOTEBOOKS.PRESENT", "fabric", "fabric", "Warning",
               "The workspace's notebooks couldn't be listed: %s" % error, doc_url=DOCS["listItems"])
else:
    names = {item.get("displayName") for item in notebooks}
    missing = [name for name in (NOTEBOOK_NAME, LIVE_ACTIVITY_NOTEBOOK) if name not in names]
    if missing:
        add_result("FABRIC.NOTEBOOKS.PRESENT", "fabric", "fabric", "Warning",
                   "Missing from the workspace: %s." % ", ".join(missing),
                   remediation="Import both notebooks with ./scripts/configure-fabric.ps1 (Fabric REST API, your own token).",
                   doc_url=DOCS["listItems"])
    else:
        add_result("FABRIC.NOTEBOOKS.PRESENT", "fabric", "fabric", "Pass",
                   "Both demo notebooks are in the workspace.", evidence={"notebooks": sorted(names & {NOTEBOOK_NAME, LIVE_ACTIVITY_NOTEBOOK})},
                   doc_url=DOCS["listItems"])

add_result("HUB.P4.TENANT_SETTING", "hub", "tenant", "Manual action required",
           "Confirm that Database Hub is enabled for your tenant (P4); no documented API reads this setting without Fabric administrator rights.",
           remediation=("Open %s. If Database Hub opens and Estate lists resources, P4 is in place. Otherwise ask a Fabric "
                        "administrator to enable Admin portal > Tenant settings > Users can access the Database hub (preview)." % DATABASE_HUB_URL),
           doc_url=DOCS["monitor"])

## 6. Cost reminders

This notebook shows **no prices**: prices change and differ by region, so the repo computes them live from the public Azure Retail Prices API.

- **Before a demo:** `./scripts/estimate-cost.ps1 -AllProfiles` prints the running cost per hour, one demo-day session, the idle month, and a light-use month for each profile (`lowest-cost`, `demo-day`, `enhanced-performance`), including any Microsoft Defender for Cloud plans on the subscription.
- **Ceilings:** the configuration sets a hard monthly ceiling (`monthlyCeilingUsd`; preflight fails above it) and a per-session ceiling (`sessionCeilingUsd`; a Warning above it). An Azure budget on the resource group alerts at 50%, 80%, and 100% of the monthly ceiling.
- **Biggest driver:** the T2 host VM size, then the SQL Server licenses that bill while T1 and the T2 guest run.
- **Bills while stopped:** managed disks, T3 storage, any Defender plans that apply to T3 or the Arc machine, the private endpoint and DNS zone in private mode, and anything left on demand, such as Bastion. Compute and SQL licenses stop when the VMs are deallocated and T3 auto-pauses.
- **Fabric:** Database Hub itself is free, but these notebooks run on this workspace's F or Trial capacity. Pause a paid capacity when you aren't using it.
- **After a demo:** `./scripts/stop-demo.ps1` (or the scheduled stop workflow) deallocates the VMs, removes Bastion, and lets T3 auto-pause; nightly auto-shutdown is the backstop. `./scripts/status.ps1` shows month-to-date spend.

## 7. Deployment status

One orchestrator, two triggers: `./scripts/deploy.ps1` (Azure Cloud Shell or local PowerShell 7, Contributor only) and the `deploy.yml` GitHub Actions workflow (after the one-time P7 setup) run the same idempotent stages. `-Resume` continues after the last good stage, recorded in the resource group tag `dbhubDeployStage`.

| Stage | What happens | How long (estimate) |
|---|---|---|
| 1 Preflight | Prerequisites P1–P6, region, sizes and quota, Azure Policy inventory, policy what-if, cost against the ceilings | Minutes |
| 2 Infrastructure | Subscription-scope Bicep: resource group, tags, budget, network, VMs, Azure SQL, auto-shutdown | About 15 minutes |
| 3 Artifacts | The script bundle reaches the VMs in SHA-256-checked Run Command chunks; installation media downloads from Microsoft with SHA-256 checks | Minutes |
| 4 Bootstraps | T1: SQL Server configuration, seed data, runner, then T3 configured from T1. T2: the host state machine builds the guest and installs SQL Server | T2 takes 60–90 minutes |
| 5 Post-configuration | T1 monitoring setting, Arc onboarding of the guest with the caller's token, Arc SQL checks, the `baseline` activity profile | Minutes |
| 6 Validation | Waits up to 15 minutes for telemetry from every target, then prints statuses, Hub links, and remaining manual items | Up to 15 minutes |
| 7 Fabric | Imports or updates both notebooks in this workspace with `./scripts/configure-fabric.ps1` | Seconds |

In the default `PaygWizard` mode, the T2 guest pauses once for a documented **manual** SQL Server setup-wizard step (pay-as-you-go Standard), then resumes on its own.

The next cell records that this notebook **can't read Azure deployment state** (no Azure Resource Manager token) and prints where to look instead.

In [ ]:
# Deployment status: Azure-side state needs an Azure Resource Manager token, which notebooks can't get. Read-only.
add_result("INFRA.AZURE.STATE", "infra", "subscription", "Not supported",
           "Fabric notebooks can't get an Azure Resource Manager token, so this notebook can't read deployment state, VM and extension health, Arc status, or Azure SQL settings.",
           remediation=("Run ./scripts/status.ps1 and ./scripts/validate.ps1 from the repo (Cloud Shell or PowerShell 7), "
                        "or open the latest deploy.yml run: %s/actions/workflows/deploy.yml" % REPO_URL),
           doc_url=DOCS["notebookutils"])

print()
print("Where to check the deployment:")
print("  Status per target, active profile, Bastion, month-to-date cost:  ./scripts/status.ps1")
print("  Full validation catalog (Azure, Arc, Azure SQL, telemetry):    ./scripts/validate.ps1")
print("  Finish or resume a deployment:                                  ./scripts/deploy.ps1 -Resume")
print("  GitHub Actions runs (optional P7):                              %s/actions" % REPO_URL)
print("  Database Hub:                                                   %s" % DATABASE_HUB_URL)

## 8. SQL Server, Azure Arc, and Azure SQL

Discovery, monitoring, agents, and mirroring are separate concerns. Each target reaches Database Hub on a different, documented path.

### T1 `sqlvm01`: SQL Server on an Azure VM

- **Image:** SQL Server 2022 Standard pay-as-you-go on Windows Server 2022. Standard (not Developer) is required: on Azure VMs, Developer, Express, and Evaluation editions send only client-connection telemetry.
- **Management path:** the SQL IaaS Agent extension. T1 is **never** Arc-enabled.
- **Estate:** the SQL IaaS Agent extension registration (and unified inventory, preview).
- **Performance prerequisites:** extension 2.0.229.0 or later, a system-assigned identity, the documented monitoring setting merged into the extension's settings, and outbound HTTPS to the telemetry endpoint for the region. `validate.ps1` checks that the extension reports `DatabaseMonitorArcPlugin` with `MetricsUploadStatus` `OK`.
- **Scenario focus:** locking and blocking, long transactions.

### T2 `arcsql01`: SQL Server enabled by Azure Arc in a nested guest

- **Disclosure:** T2 is a Hyper-V guest inside the Azure VM `archost01`, presented as a **simulated on-premises server**. The Azure Connected Machine agent and the Azure extension for SQL Server run **in the guest only**; neither the host nor the guest has the SQL IaaS extension.
- **Edition and license:** SQL Server 2022 Standard with the pay-as-you-go license type. Arc SQL performance monitoring needs Standard or Enterprise with pay-as-you-go or Software Assurance; `Developer` install mode shows T2 in Estate only. In the default `PaygWizard` mode, one documented setup-wizard step is manual per guest build.
- **Performance prerequisites:** extension 1.1.2504.99 or later, the instance's monitoring setting on, and outbound HTTPS from the guest through the host's NAT.
- **Lifecycle notes:** the guest OS is a 180-day evaluation, rebuilt with `reset-demo.ps1 -RebuildGuest`; Arc pay-as-you-go lapses after more than 30 days disconnected, and `status.ps1` warns before that.
- **Scenario focus:** CPU, memory, IO, expensive queries, missing index.

### T3 `azuresql01-<deploymentId>`: Azure SQL Database

- **Database:** `ContosoCare`, General Purpose serverless (the Azure SQL free offer when available), Microsoft Entra-only, TLS 1.2, private endpoint where tenant policy turns off public access.
- **Estate:** native, through Azure Resource Graph.
- **Performance:** the documented extended property `MS_EnablePerformanceMonitoringPreview` on the database, set from T1 during deployment. CPU on Azure SQL Database is reported as a percentage of the service tier's limit.
- **Lifecycle notes:** T3 auto-pauses when idle; the warm-up resumes it before a demo. `stop-demo.ps1 -RemoveT3` deletes it between demos to save its standing costs, and the next start recreates it with the same names.
- **Scenario focus:** service-objective CPU cap, log-rate governance, serverless resume.

Every incident scenario can also run on any target, because the Database Agent's target coverage isn't documented.

## 9. Hub telemetry validation

Database Hub **Performance** and the documented telemetry endpoint are fed by the same Microsoft-managed pipeline. The endpoint is read-only, returns only resources the signed-in user can read (Reader on the subscription, with `Microsoft.Sql` registered), keeps up to 7 days, and starts showing a newly enabled resource after a few minutes.

The telemetry article documents a Microsoft Entra token for the audience `https://kusto.kusto.windows.net`, obtained with the Azure CLI. A Fabric notebook can only ask `notebookutils` for its `kusto` audience key, and **whether the endpoint accepts that token isn't documented**. The next cell sends the article's own connectivity query and records the outcome:

- **Pass:** the endpoint accepted the notebook's token, so the checks below run here.
- **Manual action required:** it didn't (for example, HTTP 401). Every telemetry cell then prints its KQL with the documented Azure Data Explorer web UI steps, and `./scripts/validate.ps1` runs the same checks with an Azure CLI token.
- **Fail:** HTTP 403. The user lacks Reader on the subscription, or `Microsoft.Sql` isn't registered; the web UI would fail the same way.

In [ ]:
# Hub telemetry access: validates the undocumented notebook kusto-token path with the article's connectivity query. Read-only.
PROBE_KQL = """SqlServerCPUUtilization
| where SampleTimeUTC > ago(15m)
| summarize Resources = dcount(ResourceID) by ResourceType"""

probe_status, probe_frame, probe_message = _post_kql(PROBE_KQL)
TELEMETRY_STATE.update(checked=True, available=(probe_status == 200), message=probe_message)
if probe_status == 200:
    resource_types = {} if probe_frame is None or probe_frame.empty else {
        str(row["ResourceType"]): int(row["Resources"]) for _, row in probe_frame.iterrows()}
    add_result("HUB.TELEMETRY.TOKEN", "hub", "fabric", "Pass",
               "The telemetry endpoint accepted the notebook's kusto token, so this notebook validates Hub telemetry directly.",
               evidence={"httpStatus": 200, "resourcesByType": resource_types}, doc_url=DOCS["telemetry"])
elif probe_status == 403:
    add_result("HUB.TELEMETRY.TOKEN", "hub", "subscription", "Fail", probe_message,
               remediation=("Give the presenter Reader on the subscription (Contributors have it; P8 otherwise) and register "
                            "Microsoft.Sql on it (P2), then run this cell again."),
               doc_url=DOCS["telemetry"])
else:
    add_result("HUB.TELEMETRY.TOKEN", "hub", "fabric", "Manual action required",
               "The notebook can't query the telemetry endpoint directly: %s" % probe_message,
               evidence={"httpStatus": probe_status},
               remediation=("Run the KQL that the next cells print in the Azure Data Explorer web UI (%s, connection %s, database %s), "
                            "or run ./scripts/validate.ps1, which uses an Azure CLI token for https://kusto.kusto.windows.net as documented."
                            % (ADX_WEB_UI, TELEMETRY_CONNECTION_URI, TELEMETRY_DATABASE)),
               doc_url=DOCS["telemetry"])
    print_kql_fallback(PROBE_KQL, "Telemetry connectivity", probe_message)

## 10. Hub discovery and per-target validation

- **Estate (discovery)** comes from Azure Resource Graph, Hub's own inventory source. Notebooks can't query it (no Azure Resource Manager token), so the visual check is a **Manual action required** item with exact steps; `validate.ps1` runs the Resource Graph query.
- **Performance (monitoring)** is validated per target from the telemetry endpoint. The next cell lists which telemetry tables each target sent in the last `TELEMETRY_LOOKBACK_MINUTES`, filtered to `RESOURCE_GROUP` and the demo's resource names:

| Status | Meaning |
|---|---|
| **Pass** | Performance tables (CPU, waits, or performance counters) arrived within `TELEMETRY_FRESH_MINUTES` |
| **Warning** | Data stopped arriving (VM stopped, or T3 auto-paused), or only client connections arrived, which is what Developer, Express, and Evaluation editions send on Azure VMs |
| **Fail** | No telemetry at all in the window: the target is stopped, or a monitoring prerequisite is missing |
| **Not supported** | T2 in `Developer` install mode: Arc SQL performance monitoring needs Standard or Enterprise |
| **Not selected** | The target isn't in `TARGETS` |
| **Manual action required** | The notebook couldn't query the endpoint; run the printed KQL |

A newly started or recreated target can take 3–15 minutes to report. An empty result never means a resource is healthy: it only means nothing was received.

In [ ]:
# Per-target validation: telemetry tables per target in the lookback window, plus the Estate check. Read-only.
window_end = utc_now()
window_start = window_end - dt.timedelta(minutes=int(TELEMETRY_LOOKBACK_MINUTES))
COVERAGE_WINDOW.update(start=window_start, end=window_end)
COVERAGE_KQL = kql_prelude(window_start, window_end) + """
union withsource=TableName SqlServer*
| where SampleTimeUTC between (_start .. _end)
| where ResourceGroup =~ _rg
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| summarize Rows = count(), LastSampleUTC = max(SampleTimeUTC), Instances = dcount(SqlServerInstanceName)
        by Target, ResourceType, ResourceName, TableName
| order by Target asc, TableName asc
| take 500"""

COVERAGE = query_telemetry(COVERAGE_KQL, "Telemetry tables per target")
PERFORMANCE_TABLES = {"SqlServerCPUUtilization", "SqlServerWaitStats", "SqlServerPerformanceCountersCommon"}
STALE_HINTS = {"T1": "is the VM stopped or deallocated?", "T2": "is the host VM stopped or deallocated?", "T3": "is the database auto-paused?"}
PARTIAL_TELEMETRY_HINTS = {
    "T1": "On Azure VMs, Developer, Express, and Evaluation editions send only client connections; T1 needs Standard or Enterprise.",
    "T2": "Arc SQL performance monitoring needs Standard or Enterprise with pay-as-you-go or Software Assurance, and monitoring enabled.",
    "T3": "Check that performance monitoring is enabled on the database.",
}
MISSING_TELEMETRY_STEPS = {
    "T1": ("Start T1 (./scripts/start-demo.ps1 or demo-control.yml), wait 3-15 minutes, and run this cell again. If it stays empty, "
           "run ./scripts/validate.ps1: the SQL IaaS Agent extension must report DatabaseMonitorArcPlugin with MetricsUploadStatus OK."),
    "T2": ("Start the T2 host (the guest starts with it), wait 3-15 minutes, and run this cell again. If it stays empty, run "
           "./scripts/validate.ps1: the Arc machine must be Connected, the SQL extension 1.1.2504.99 or later, and monitoring enabled."),
    "T3": ("Resume T3 with the warm-up (./scripts/start-demo.ps1) or any query, wait 3-15 minutes, and run this cell again. If it stays "
           "empty, run ./scripts/validate.ps1: MS_EnablePerformanceMonitoringPreview must be set on %s." % DEMO_DATABASE),
}

for target in ("T1", "T2", "T3"):
    check_id = "HUB.TELEMETRY." + target
    area = target.lower()
    if target not in SELECTED_TARGETS:
        add_result(check_id, area, target, "Not selected", "%s isn't in TARGETS." % TARGET_NAMES[target])
    elif target == "T2" and T2_SQL_INSTALL_MODE == "Developer":
        add_result(check_id, area, target, "Not supported",
                   "T2 runs SQL Server Developer edition: it appears in Estate, but Arc SQL performance monitoring needs Standard or Enterprise with pay-as-you-go or Software Assurance.",
                   doc_url=DOCS["telemetry"])
    elif COVERAGE is None:
        add_result(check_id, area, target, "Manual action required",
                   "Telemetry for %s wasn't queried from the notebook." % TARGET_NAMES[target],
                   remediation="Run the printed KQL in the Azure Data Explorer web UI, or run ./scripts/validate.ps1.",
                   doc_url=DOCS["telemetry"])
    else:
        rows = COVERAGE[COVERAGE["Target"] == target] if "Target" in COVERAGE.columns else COVERAGE
        if rows.empty:
            add_result(check_id, area, target, "Fail",
                       "No telemetry from %s in the last %s minutes." % (TARGET_NAMES[target], TELEMETRY_LOOKBACK_MINUTES),
                       remediation=MISSING_TELEMETRY_STEPS[target], doc_url=DOCS["addSql"])
            continue
        tables = sorted(rows["TableName"].unique())
        last_sample = rows["LastSampleUTC"].max()
        age_minutes = None if pd.isna(last_sample) else round((window_end - last_sample).total_seconds() / 60, 1)
        evidence = {"resourceName": rows["ResourceName"].iloc[0], "tables": tables, "rows": int(rows["Rows"].sum()),
                    "lastSampleUtc": None if pd.isna(last_sample) else last_sample.isoformat(), "lastSampleAgeMinutes": age_minutes}
        if not PERFORMANCE_TABLES.intersection(tables):
            add_result(check_id, area, target, "Warning",
                       "%s sends only %s, not performance data. %s" % (TARGET_NAMES[target], ", ".join(tables), PARTIAL_TELEMETRY_HINTS[target]),
                       evidence=evidence, remediation=MISSING_TELEMETRY_STEPS[target], doc_url=DOCS["telemetry"])
        elif age_minutes is None or age_minutes > int(TELEMETRY_FRESH_MINUTES):
            add_result(check_id, area, target, "Warning",
                       "%s stopped reporting %s minutes ago (%s)." % (TARGET_NAMES[target], "an unknown number of" if age_minutes is None else age_minutes, STALE_HINTS[target]),
                       evidence=evidence, remediation=MISSING_TELEMETRY_STEPS[target], doc_url=DOCS["telemetry"])
        else:
            add_result(check_id, area, target, "Pass",
                       "%s reports performance telemetry from %d tables; the last sample is %s minutes old." % (TARGET_NAMES[target], len(tables), age_minutes),
                       evidence=evidence, doc_url=DOCS["telemetry"])

selected_names = ", ".join(TARGET_NAMES[target] for target in SELECTED_TARGETS)
add_result("HUB.ESTATE.VISUAL", "hub", "subscription", "Manual action required",
           "Confirm in Database Hub that the selected targets appear in Estate: %s." % selected_names,
           remediation=("Open %s > Estate, set View to Microsoft SQL, filter to resource group %s, and look for each target. "
                        "Estate comes from Azure Resource Graph, which notebooks can't query; ./scripts/validate.ps1 runs that query."
                        % (DATABASE_HUB_URL, RESOURCE_GROUP)),
           doc_url=DOCS["monitor"])

if COVERAGE is not None and not COVERAGE.empty:
    view = COVERAGE.copy()
    view["LastSampleAgeMinutes"] = ((window_end - view["LastSampleUTC"]).dt.total_seconds() / 60).round(1)
    display_frame(view[["Target", "ResourceType", "ResourceName", "TableName", "Rows", "Instances", "LastSampleAgeMinutes"]])

## 11. Agent skills and the Database Agent

**The showcase: Database Hub's in-Hub Database Agent (preview).** Agent findings appear in **Overview** and **Estate** (issue details with a summary and a recommended action), and in the conversational Copilot or agent surface where your tenant shows it. Copilot is the conversational interface; agents are the analytical and operational engines, and an action is backed by an auditable proposal.

- It's **presenter-driven, in the Hub UI**. There's no public API, so nothing here automates, scrapes, or calls it.
- Any proposed action needs **explicit human approval** before anything runs.
- Its supported targets, prerequisites, and latency aren't documented yet, so every incident scenario can run on T1, T2, or T3, and the walkthrough uses whichever targets the agent covers in your tenant.

**The documented, scriptable fallback** (also used when the agent isn't available):

| Piece | Role |
|---|---|
| GitHub Copilot CLI or VS Code | Agent host |
| `microsoft-sql-fdh` plugin (Database Hub agent skills) | Hub inventory, performance health, and security posture; read-only, with your delegated sign-in |
| `microsoft-sql` plugin | Blocking, deadlocks, resource pressure, slow queries, and plans, run against T1 or T3 from an allowed client |
| Telemetry article's agent instructions | Investigating the same telemetry with KQL, with attribution to Microsoft Learn |
| Microsoft Learn MCP server | Grounding answers in the official docs |

The prompt pack, with the surface, data source, permission, response shape, preview limitation, and fallback for each prompt, is in `fabric/agent/` in the repo. The Fabric data agent is out of scope: it works over OneLake data, not Hub signals.

## 12. Optional mirroring

**Not part of this demo (deferred from v1).** Database Hub doesn't need Fabric mirroring: it monitors databases in place. Mirroring would add capacity, network, SQL Server version, and permission requirements (for example, Azure SQL Database mirroring needs vCore or at least 100 DTUs and a server-assigned managed identity, and SQL Server 2025 mirroring requires Azure Arc). It's documented as a possible future add-on in `docs/adr/0015-mirroring-deferral.md`, `docs/faq.md`, and `docs/limitations.md`.

## 13. Data generation and scenarios

**Data.** Every target hosts the database `ContosoCare` for the fictional "Contoso Care Operations": obviously synthetic facilities, providers, patients, appointments, encounters, claims, orders, inventory, payments, audit events, and work queues. No real or personal data. A deterministic, set-based seed builds the same data on every target.

**Workload.** The .NET runner `ContosoCare.Workload` runs on T1 (for T1 and T3) and in the T2 guest. It tags every connection with the application name `ContosoCare.Demo/<scenarioId>/<runId>`, enforces a hard maximum duration, and only ever touches its own sessions inside `ContosoCare`. A watchdog rolls back anything that outlives its scenario.

| Scenario | Default target | What it does | Hub telemetry evidence | Query-level evidence |
|---|---|---|---|---|
| Healthy baseline (`baseline` profile) | Any | A steady OLTP mix: the "before" picture | Steady CPU, batch requests, sessions | Query Store baseline |
| Locking and blocking (`incident:locking-blocking`) | T1 | One session holds locks for a bounded time while others block; optional long transaction | `Processes blocked`, Lock-category waits, `Active Transactions`, `SqlServerActiveSessions` | Blocking chain in SSMS or the `diagnose-blocking-and-deadlocks` skill |
| CPU pressure (`incident:cpu-pressure`) | T2 | A bounded expensive query at configurable concurrency, plus an indexed healthy version | CPU utilization, CPU and signal waits | Query Store before and after |
| IO pressure (`incident:io-pressure`) | T2 | Bounded reads and writes over seeded data; checks free space first | `SqlServerStorageIO` volume and stalls, `PAGEIOLATCH` (Buffer IO) waits | `sys.dm_io_virtual_file_stats` |
| Query optimization (`incident:inefficient-query`) | T2, T3 | A repeatable plan problem; on T3, log-rate pressure | CPU (on T3, as a percentage of the tier limit), `Log Rate Governor` waits | Query Store regressed query |
| Missing index (`incident:missing-index`) | T2 | A query that a missing index would fix: before, recommendation, action, after | CPU and Buffer IO waits | Missing-index DMVs and Query Store; the fix is never applied automatically |

**Activity profiles** combine them: `baseline`, `business-day` (the baseline plus scheduled bounded incidents on each target), `incident:<scenario>`, and `off`. Every profile has a maximum duration and stops at auto-shutdown.

**How to run them:** `NB_DatabaseHub_LiveActivity` (it can trigger the `demo-control.yml` workflow when P9 is set up, and otherwise prints the link and command), the GitHub Actions workflow itself, or `./scripts/start-demo.ps1` from the repo.

## 14. Hub investigation, Database Agent prompts, and expected findings

**Where to look in Database Hub** ([open Database Hub](https://powerbi.com/workloads/fdh/databaseHub)):

1. **Overview:** **What needs attention?** Every card leads to Estate.
2. **Estate:** set **View** to **Microsoft SQL**, filter to the demo resource group, and select a target. Issues appear as colored chips and suggestions as lightbulb chips.
3. **Performance:** the **Microsoft SQL** dashboard. Check the resource selection before you treat a chart as the whole estate.
4. **Database Agent and Copilot:** ask the prompts below about a finding or a target.

**Prompts to try.** Describe the signals to look for, never a specific answer: the agent's output isn't deterministic, and its coverage is a preview.

| Prompt | Signals to look for in the response | If the agent doesn't cover it |
|---|---|---|
| "Summarize the health of my SQL estate in resource group `rg-dbhub-demo`. What needs attention first?" | All selected targets named; open issues and suggestions ranked | Overview and Estate pages; the `microsoft-sql-fdh` skills |
| "Which of my SQL resources is under the most pressure right now, and why?" | CPU, waits, or blocking tied to a specific target and time window | Performance page; live-activity notebook, sections 5–7 |
| "Investigate blocking on `sqlvm01` in the last 30 minutes." | `Processes blocked` above 0, Lock waits, long-running transactions | The `diagnose-blocking-and-deadlocks` skill; live-activity notebook, section 6 |
| "Correlate it with CPU, memory, IO, sessions, and recent changes." | A timeline across several signals, not just one | Live-activity notebook, sections 6–8 |
| "Explain the root cause in customer-friendly terms, with the supporting evidence." | Plain language, with the metrics and time window it relied on | Presenter narration from the charts |
| "What's the safest next action? Don't apply it." | A proposal that isn't executed, and that names what needs approval | Remediation scripts in the repo, reviewed by a person |
| "Compare the baseline with the incident." | Before and after values for the same signals | Live-activity notebook, section 8 (correlation) |
| "Which queries are inefficient on `arcsql01`?" | A pointer to Query Store: the telemetry describes resources, not individual queries | Query Store reports or the `diagnose-slow-query` skill |
| "Which of these actions need approval?" | Every change listed as needing a person to approve it | ADR 0013 and the walkthrough |
| "What should I verify in the Azure portal, SSMS, or VS Code?" | Concrete places to confirm each finding | Presenter's checklist |
| "Write a concise incident summary." | Timeline, impact, cause, action, and verification | Presenter's summary |

**Expected findings are signals, not text.** During a locking incident on T1, expect `Processes blocked` above 0 and Lock waits in that window; during CPU pressure on T2, sustained CPU and higher signal waits; during IO pressure, `PAGEIOLATCH` waits and read volume; on T3, CPU near the tier limit or `Log Rate Governor` waits. Telemetry arrives 3–15 minutes after the activity.

## 15. Remediation, reset and rerun, start and stop

**Remediation is always a human decision.** The Database Agent and the skills propose; a person approves. Remediation scripts in the repo, such as the missing-index fix, are **never applied automatically**. For blocking, the safe action is to stop the scenario (or let its watchdog roll it back), which ends the blocking session inside `ContosoCare` only.

**Reset and rerun** (no redeploy):

- `./scripts/reset-demo.ps1` stops scenarios, verifies that no demo transaction is still open, and restores the seed state.
- `./scripts/reset-demo.ps1 -RebuildGuest` rebuilds the T2 guest, for example before its evaluation OS expires.
- Then start a profile again from `NB_DatabaseHub_LiveActivity`, the `demo-control.yml` workflow, or `./scripts/start-demo.ps1`.

**Start and stop:**

| Action | Command | Notes |
|---|---|---|
| Start | `./scripts/start-demo.ps1` | Starts T1 and the T2 host (the guest starts with it), warms up T3, and sets an activity profile. `-Interactive` creates Bastion on demand. |
| Stop | `./scripts/stop-demo.ps1` | Stops the profile, shuts the T2 guest down cleanly before deallocating the host, deallocates T1, lets T3 auto-pause, and removes Bastion. `-ParkT2` stops only T2; `-RemoveT3` deletes T3 until the next start. |
| Status | `./scripts/status.ps1` | Health per target, active profile, evaluation-OS days left, Arc days disconnected, Bastion, month-to-date cost |
| GitHub Actions | `demo-control.yml`, scheduled start and stop workflows | Optional P7; destructive workflows need approval |

## 16. Validation summary

The next cell shows every result of this run and, when `SAVE_RESULTS = True`, appends them to the Delta table `validation_results` and the telemetry table coverage to `telemetry_snapshots` in the default lakehouse. Writing these tables changes only lakehouse data, not Azure or Fabric resources. Re-running the cell in the same session replaces that run's rows instead of duplicating them.

Success means every selected capability is **Pass** or is explicitly **Manual action required** or **Not supported**. A successful deployment alone is never reported as **Pass**.

In [ ]:
# Validation summary, then save to validation_results and telemetry_snapshots (lakehouse data only).
summary = show_results()
failures = [item["check_id"] for item in RESULTS if item["status"] == "Fail"]
manual = [item["check_id"] for item in RESULTS if item["status"] == "Manual action required"]
print("Fail: %s" % (", ".join(failures) if failures else "none"))
print("Manual action required: %s" % (", ".join(manual) if manual else "none"))

if SAVE_RESULTS:
    write_delta("validation_results", RESULTS)
    snapshot_rows = []
    if COVERAGE is not None and not COVERAGE.empty:
        captured = utc_now()
        for _, row in COVERAGE.iterrows():
            base = {"run_id": RUN_ID, "source_notebook": NOTEBOOK_NAME, "captured_at_utc": captured, "analysis": "telemetry_coverage",
                    "target": row["Target"], "resource_type": row["ResourceType"], "resource_name": row["ResourceName"],
                    "instance_name": None, "database_name": None, "bucket_utc": row["LastSampleUTC"],
                    "window_start_utc": COVERAGE_WINDOW["start"], "window_end_utc": COVERAGE_WINDOW["end"]}
            snapshot_rows.append(dict(base, metric=row["TableName"] + ".rows", value=float(row["Rows"]), unit="rows"))
            snapshot_rows.append(dict(base, metric=row["TableName"] + ".instances", value=float(row["Instances"]), unit="instances"))
    write_delta("telemetry_snapshots", snapshot_rows)
else:
    print("SAVE_RESULTS is False, so nothing was saved.")

## 17. Troubleshooting

| Symptom | Likely cause | What to do |
|---|---|---|
| `FABRIC.P5.CAPACITY` is **Fail** | The workspace isn't on F or Trial capacity | Assign a capacity in workspace settings; Database Hub stays free, but notebooks need capacity |
| `FABRIC.SESSION.LAKEHOUSE` is **Fail** | No default lakehouse | Add `LH_DatabaseHub` in the Explorer pane and set it as the default |
| `HUB.TELEMETRY.TOKEN` is **Manual action required** (HTTP 401) | The endpoint doesn't accept the notebook's `kusto` token (undocumented path) | Run the printed KQL in the Azure Data Explorer web UI, or `./scripts/validate.ps1` |
| `HUB.TELEMETRY.TOKEN` is **Fail** (HTTP 403) | No Reader on the subscription, or `Microsoft.Sql` isn't registered | Assign Reader (P8) and register `Microsoft.Sql` (preflight does it with Contributor) |
| A target is **Fail** with no telemetry | The VM is stopped, T3 is paused, or monitoring isn't enabled | Start the demo, wait 3–15 minutes, then run `./scripts/validate.ps1` for the target's monitoring prerequisites |
| T1 sends only client connections | Developer, Express, or Evaluation edition | Use the Standard image (the default) |
| T2 is missing from Estate | The guest isn't Arc-onboarded yet; in `PaygWizard` mode, the setup-wizard step may be pending | Check `./scripts/status.ps1`; finish the manual step, then `./scripts/deploy.ps1 -Resume` |
| T3 telemetry stops | Serverless auto-pause | Expected when idle; the warm-up resumes it |
| Numbers look far too large in your own KQL | Cumulative counters were summed | Take deltas between samples (telemetry article, rule 2) |
| A Hub finding doesn't clear | Signals refresh with a delay | Confirm the change completed, wait, and refresh Database Hub |
| A recommendation conflicts with the evidence | Scope or time range mismatch | Don't apply it. Recheck the resource, the time range, and the approval path |

## 18. Cleanup

**Azure resources** are removed from the repo, never from this notebook:

- `./scripts/destroy.ps1` (or the `destroy.yml` workflow, which needs approval) deletes the Arc SQL instance and the Arc machine explicitly, then the resource group, then checks with Azure Resource Graph that nothing tagged with the deployment ID remains.
- Between demos, `./scripts/stop-demo.ps1` is enough: nothing runs, and only storage and any Defender plans keep billing. `-RemoveT3` also removes T3 until the next start.

**Lakehouse history** stays after a destroy: `validation_results`, `scenario_runs`, `telemetry_snapshots`, `analysis_findings`, and `incident_windows` keep demo history beyond the telemetry's 7-day retention. The Power BI history report (`RPT_DatabaseHub_History`, model `SM_DatabaseHub_History`) reads all five.

> **Resource-changing cell.** The next cell **drops those five Delta tables** from the default lakehouse. The report's Direct Lake model can't refresh again until `NB_DatabaseHub_LiveActivity` runs with `SAVE_RESULTS = True` and recreates them. It's destructive and can't be undone, so it does nothing unless you set both `PURGE_DEMO_TABLES = True` and `CONFIRM = True` in the parameters cell. It never touches Azure resources.

In [ ]:
# RESOURCE-CHANGING CELL: drops the demo's Delta tables from the default lakehouse. Destructive; needs PURGE_DEMO_TABLES and CONFIRM.
DEMO_TABLES = ("validation_results", "scenario_runs", "telemetry_snapshots", "analysis_findings", "incident_windows")
MODEL_NOTE = ("The Power BI history report's semantic model (SM_DatabaseHub_History, Direct Lake) can't refresh "
              "until NB_DatabaseHub_LiveActivity runs with SAVE_RESULTS = True and recreates the tables.")


def drop_demo_tables():
    """Drops the demo's Delta tables. Destructive: refuses to run unless CONFIRM and PURGE_DEMO_TABLES are True."""
    if not (CONFIRM and PURGE_DEMO_TABLES):
        raise PermissionError("CONFIRM and PURGE_DEMO_TABLES must both be True. Nothing was dropped.")
    session = _spark()
    if session is None:
        print("Spark isn't available, so nothing was dropped.")
        return []
    dropped = []
    for table in DEMO_TABLES:
        if session.catalog.tableExists(table):
            session.sql("DROP TABLE IF EXISTS %s" % table)
            dropped.append(table)
    return dropped


if not (CONFIRM and PURGE_DEMO_TABLES):
    print("Dry run: this cell needs both PURGE_DEMO_TABLES = True and CONFIRM = True, so no table was dropped. "
          "Tables it would drop: %s." % ", ".join(DEMO_TABLES))
    print("After a drop: " + MODEL_NOTE)
else:
    removed = drop_demo_tables()
    print("Dropped: %s." % (", ".join(removed) if removed else "nothing (no demo tables found)"))
    if removed:
        print(MODEL_NOTE)

## 19. Limitations and preview disclosures

- **Previews:** Database Hub, the Database Agent and Copilot, SQL Server on Azure VM performance monitoring and unified inventory, the Arc SQL monitoring setting, and the telemetry endpoint can change without notice. Check the official docs before a customer demo.
- **Database Agent:** presenter-driven, no API, and undocumented target coverage, prerequisites, and latency. Never promise a specific answer.
- **Notebook boundaries:** no Azure Resource Manager token, no line of sight to the private SQL Servers, and an undocumented `kusto` token path with a documented fallback.
- **Telemetry:** retention of up to 7 days, 3–15 minutes of latency, resources without monitoring don't appear at all, and the data describes resources, not individual queries (use Query Store for query text and plans). Per-file IO latency isn't supported for Azure SQL Database.
- **Editions:** Developer, Express, and Evaluation editions on Azure VMs send only client connections; Arc SQL monitoring needs Standard or Enterprise with pay-as-you-go or Software Assurance.
- **T2 is simulated on-premises:** a nested Hyper-V guest on an Azure VM, with a 180-day evaluation OS and, in `PaygWizard` mode, one manual setup-wizard step per guest build. Arc pay-as-you-go lapses after more than 30 days disconnected.
- **Manual by design:** the Hub tenant setting (P4), Estate's visual check, Hub saved views and monitor templates, and every Database Agent interaction.
- **Not included:** Fabric mirroring, the Fabric data agent, and Azure SQL Managed Instance.

The repo's `docs/limitations.md` has the complete list.

## 20. Official references

**Database Hub**

- Overview: <https://learn.microsoft.com/fabric/database/hub/overview>
- Monitor Microsoft SQL: <https://learn.microsoft.com/fabric/database/hub/monitor-sql>
- Add Microsoft SQL resources (enable performance monitoring): <https://learn.microsoft.com/fabric/database/hub/add-sql>
- FAQ: <https://learn.microsoft.com/fabric/database/hub/faq>
- Query performance monitoring telemetry (endpoint, schema, rules, agent instructions): <https://learn.microsoft.com/fabric/database/hub/microsoft-sql-query-performance-monitoring-telemetry>
- Database Hub agent skills (`microsoft-sql-fdh`): <https://github.com/microsoft/microsoft-sql/tree/main/plugins/microsoft-sql-fdh>

**Fabric notebooks and REST API**

- NotebookUtils credentials (`getToken` audience keys, `getSecret`): <https://learn.microsoft.com/fabric/data-engineering/notebookutils/notebookutils-credentials>
- NotebookUtils runtime context: <https://learn.microsoft.com/fabric/data-engineering/notebookutils/notebookutils-runtime>
- Develop and run notebooks (parameter cells): <https://learn.microsoft.com/fabric/data-engineering/author-execute-notebook>
- Workspaces - Get Workspace: <https://learn.microsoft.com/rest/api/fabric/core/workspaces/get-workspace>
- Items - Get Lakehouse: <https://learn.microsoft.com/rest/api/fabric/lakehouse/items/get-lakehouse>
- Items - List Items: <https://learn.microsoft.com/rest/api/fabric/core/items/list-items>
- Fabric licenses (notebooks need capacity): <https://learn.microsoft.com/fabric/enterprise/licenses>

**SQL targets**

- SQL Server on Azure VMs performance monitoring: <https://learn.microsoft.com/azure/azure-sql/virtual-machines/windows/enable-performance-monitoring-sql-vm>
- SQL Server enabled by Azure Arc monitoring: <https://learn.microsoft.com/sql/sql-server/azure-arc/sql-monitoring>
- Azure SQL Database serverless auto-pause: <https://learn.microsoft.com/azure/azure-sql/database/serverless-tier-auto-pause-resume>
- Query Store: <https://learn.microsoft.com/sql/relational-databases/performance/monitoring-performance-by-using-the-query-store>

## 21. Next steps

1. Fix anything that's **Fail**, and work through the **Manual action required** items in the summary (P4 and the Estate visual check are always manual).
2. Open **`NB_DatabaseHub_LiveActivity`** to start an activity profile (or print the link and command for it) and to analyze locks, blocking, and consumption with scenario windows overlaid.
3. Open [Database Hub](https://powerbi.com/workloads/fdh/databaseHub) and walk Overview, Estate, and Performance, then try the Database Agent prompts in section 14.
4. After the demo, stop the compute with `./scripts/stop-demo.ps1` (or let nightly auto-shutdown do it), and re-run this notebook next time to compare `validation_results` across runs.